# 03. Stack-aware CI 정책

## 학습 목표

- github.event.pull_request.stack metadata를 Python 구조로 모델링합니다.
- 모든 layer, top, lowest unmerged에서 실행할 job을 구분합니다.
- stack 길이에 따른 CI 비용 증가와 최적화 trade-off를 계산합니다.

GitHub Actions expression을 실제로 실행하지 않는 policy simulator입니다.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class StackMeta:
    number: int
    size: int
    position: int
    stack_base_ref: str
    pull_request_base_ref: str

    @property
    def is_top(self) -> bool:
        return self.position == self.size

    @property
    def is_original_bottom(self) -> bool:
        return self.position == 1

    @property
    def is_lowest_unmerged(self) -> bool:
        # 공식 문서 조건: stack base와 현재 PR base가 같음
        return self.stack_base_ref == self.pull_request_base_ref

metas = [
    StackMeta(42, 4, 1, "main", "main"),
    StackMeta(42, 4, 2, "main", "layer-1"),
    StackMeta(42, 4, 3, "main", "layer-2"),
    StackMeta(42, 4, 4, "main", "layer-3"),
]

In [ ]:
def should_run(policy: str, meta: StackMeta | None) -> bool:
    """non-stack PR에서는 필수 job을 안전하게 실행합니다."""
    if meta is None:
        return policy in {"all", "top", "lowest"}
    if policy == "all":
        return True
    if policy == "top":
        return meta.is_top
    if policy == "lowest":
        return meta.is_lowest_unmerged
    if policy == "original-bottom":
        return meta.is_original_bottom
    if policy == "release-base":
        return meta.stack_base_ref.startswith("release/")
    raise ValueError(f"알 수 없는 policy: {policy}")

for meta in metas:
    print(
        f"position {meta.position}/{meta.size}",
        {name: should_run(name, meta) for name in ("all", "top", "lowest", "original-bottom")},
    )

assert should_run("top", metas[-1])
assert should_run("lowest", metas[0])

In [ ]:
jobs = {
    "lint": ("all", 2),
    "unit-test": ("all", 6),
    "security-scan": ("all", 4),
    "integration-test": ("top", 20),
    "base-compatibility": ("lowest", 10),
}

def planned_jobs(meta: StackMeta | None) -> list[str]:
    return [name for name, (policy, _) in jobs.items() if should_run(policy, meta)]

total_minutes = 0
for meta in metas:
    selected = planned_jobs(meta)
    cost = sum(jobs[name][1] for name in selected)
    total_minutes += cost
    print(f"PR {meta.position}: {selected} -> {cost} runner-minutes")
print("stack 전체 예상 비용:", total_minutes, "runner-minutes")

In [ ]:
def compare_cost(stack_size: int) -> tuple[int, int]:
    all_jobs_each_layer = stack_size * sum(minutes for _, minutes in jobs.values())
    optimized = stack_size * sum(
        minutes for policy, minutes in jobs.values() if policy == "all"
    )
    optimized += sum(minutes for policy, minutes in jobs.values() if policy in {"top", "lowest"})
    return all_jobs_each_layer, optimized

for size in (1, 2, 4, 8):
    naive, optimized = compare_cost(size)
    print(f"layers={size}: naive={naive:3d}, optimized={optimized:3d}, saved={naive-optimized:3d}")

# 빠른 quality gate는 모든 layer에 남겨 둡니다.
assert planned_jobs(metas[1]) == ["lint", "unit-test", "security-scan"]

## GitHub Actions로 옮길 때

top layer 조건은 stack이 null이 아니고 position과 size가 같은지 검사합니다. 실제 workflow에서는 non-stack PR 경로도 별도로 포함합니다. 보안 scan이나 빠른 unit test를 top에만 실행하면 lower layer 결함이 늦게 발견될 수 있으므로 expensive integration job에만 위치 조건을 적용하는 것이 안전합니다.

## 확장 과제

1. changed path에 따라 expensive job을 추가로 줄이는 조건을 만듭니다.
2. bottom merge 후 position은 같지만 lowest unmerged가 바뀌는 상황을 시뮬레이션합니다.
3. merge queue에서 lower PR이 eject될 때 위 PR job을 취소하는 정책을 설계합니다.
4. stack 크기와 review latency, CI 비용을 함께 최적화하는 최대 layer 수를 계산합니다.